# V54 - CatBoost YetiRank as a champion-scale base view

Pre-registered hypothesis: the Ridge views plateau near 0.62 because pointwise regression is the wrong objective for NDCG. A listwise ranker (CatBoost YetiRank, the same objective family as the v5/v6 and v42 lineage) over identical features should clear +0.005 pooled OOF against the chat-Ridge baseline with no negative fold. If it does, it becomes the base view that future slice actions compose onto. If not, the Ridge plateau is real and the road continues only via the full heavy stack.

In [ ]:
import json
import re
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRanker, Pool
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 99
N_SPLITS = 5
BASE_FLOOR = 0.005
ITERS, DEPTH, LR = 300, 6, 0.05
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v54")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve(), "| OUT:", OUT.resolve())

## Features and listwise OOF

Identical combined features for both contenders, so the comparison isolates the objective (listwise YetiRank vs pointwise Ridge). Query id is the user. Folds stay template-grouped under a fresh seed.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)

by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
all_users = list(train["user_id"]) + test_ids
blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan

A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
T = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3).fit_transform(docs).toarray()
X = np.hstack([T, A]).astype(np.float32)
X_tr, X_te = X[:n_train], X[n_train:]
print("combined features:", X.shape)

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

gid_tr = np.repeat(np.arange(n_train), len(MODS))
Xl_tr = np.repeat(X_tr, len(MODS), axis=0)
yl_tr = np.rint(Y.reshape(-1) * 5).astype(int)
gid_te = np.repeat(np.arange(len(test_ids)), len(MODS))
Xl_te = np.repeat(X_te, len(MODS), axis=0)

oof_ridge = np.zeros_like(Y)
pred_ridge = np.zeros((len(test_ids), len(MODS)))
oof_cb = np.zeros_like(Y)
pred_cb = np.zeros((len(test_ids), len(MODS)))
fid = np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(X_tr, strata, fold_groups)):
    for j in range(len(MODS)):
        m = Ridge(alpha=3.0, random_state=SEED)
        m.fit(X_tr[tr], Y[tr, j])
        oof_ridge[va, j] = m.predict(X_tr[va])
        pred_ridge[:, j] += m.predict(X_te) / N_SPLITS
    trm = np.isin(gid_tr, tr)
    vam = np.isin(gid_tr, va)
    pool_tr = Pool(Xl_tr[trm], yl_tr[trm], group_id=gid_tr[trm])
    pool_va = Pool(Xl_tr[vam], yl_tr[vam], group_id=gid_tr[vam])
    cb = CatBoostRanker(iterations=ITERS, depth=DEPTH, learning_rate=LR, loss_function="YetiRank",
                        random_seed=SEED, verbose=False, allow_writing_files=False)
    cb.fit(pool_tr)
    oof_cb[va] = cb.predict(pool_va).reshape(len(va), len(MODS))
    pred_cb += cb.predict(Pool(Xl_te, group_id=gid_te)).reshape(len(test_ids), len(MODS)) / N_SPLITS
    fid[va] = fi + 1
    print(f"fold {fi + 1}: ridge={ndcg5(Y[va], oof_ridge[va]).mean():.5f} catboost={ndcg5(Y[va], oof_cb[va]).mean():.5f}", flush=True)

## Gate the ranker and write the new base

Adoption needs a pooled CI lower bound above +0.005 with no negative fold. On success the CatBoost test predictions are saved as the base view for future actions. Otherwise Ridge stays and the heavy stack remains the only road.

In [ ]:
base = ndcg5(Y, oof_ridge)
d = ndcg5(Y, oof_cb) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fid == k], oof_cb[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"pooled ridge={base.mean():.5f} catboost={(base + d).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
adopt = (lo > BASE_FLOOR) and (wf >= 0)
print("ADOPT CatBoost as base view" if adopt else "KEEP Ridge; ranker objective alone does not lift this feature set")
np.save(OUT / ("base_catboost_test.npy" if adopt else "candidate_catboost_test.npy"), pred_cb)
np.save(OUT / "base_ridge_test.npy", pred_ridge)
test_ids_s = pd.Series(test_ids)
test_ids_s.to_csv(OUT / "test_ids.csv", index=False)
print("saved")